# PWM-Dyn v0.5: validation sweep + 3-seed training + ParadoxBench-1 scoring (Kaggle, GPU T4 x2)
Self-contained; last cell runs `run_v05.py` (~2.5 h).

In [ ]:
import hashlib, os
import torch; print("GPUs:", torch.cuda.device_count()); assert torch.cuda.is_available(), "set Accelerator to GPU T4 x2"
os.makedirs("/kaggle/working/pwm_kit/data/raw", exist_ok=True)
os.makedirs("/kaggle/working/pwm_kit/pwm", exist_ok=True)
open("/kaggle/working/pwm_kit/pwm/__init__.py","w").write(r'''''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/pwm/__init__.py").read().encode()).hexdigest()[:16] == "e3b0c44298fc1c14", "typo in pwm/__init__.py"
open("/kaggle/working/pwm_kit/pwm/data.py","w").write(r'''"""PWM data layer: real physical-world observations on a spatial graph.

Every dataset becomes the same structure:
  raw     [T, N]  observations in real units (NaN = missing / not observed)
  A       [N, N]  row-normalised physical adjacency (road distances or geographic grid)
  period  index of the observation inside its natural cycle (5-min slot of day / day of year)
  dow     day of week (0 for daily climate data, where it carries no meaning)

Datasets
  PEMS08, PEMS04   Caltrans road-sensor traffic flow, 5-min steps (standard benchmark protocol:
                   12 -> 12 steps, chronological 60/20/20 split, zero-masked metrics)
  IMD_<region>     India Meteorological Department 0.25 deg gridded daily rainfall (mm/day),
                   30 days -> next 7 days, split by year, zeros are REAL (no zero-masking)
"""
import datetime as dt
import os

import numpy as np
import torch

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RAW = os.path.join(ROOT, "data", "raw")

# lat_min, lat_max, lon_min, lon_max  (0.25 deg IMD grid)
IMD_REGIONS = {
    "central_india": (21.0, 26.75, 74.0, 82.75),   # Madhya Pradesh and surroundings
    "north_india":   (26.0, 31.75, 74.0, 80.75),   # Delhi NCR, Haryana, west UP, Punjab
    "west_coast":    (14.0, 20.75, 72.5, 76.25),   # Mumbai, Konkan, Western Ghats
}


def _row_norm(A):
    np.fill_diagonal(A, 1.0)
    return torch.tensor(A / A.sum(1, keepdims=True), dtype=torch.float32)


class SeriesData:
    """lag > 0 (v0.4 long-context variant): each window also carries the values from one season
    earlier (lag steps back: same time yesterday for traffic, same days last year for rainfall),
    covering lag_pre steps before and the whole forecast horizon. Total look-back = lag + lag_pre,
    which is kept within the context TimesFM is given (512 steps / 365 days).
    stats = (mean, std) overrides train-split normalisation (zero-shot transfer: source stats)."""

    def __init__(self, name, raw, A, period_idx, dow, period, in_len, out_len, bounds,
                 transform="none", mask_zero=False, units="", meta=None, lag=0, lag_pre=0, stats=None,
                 profile=None):
        self.name, self.raw, self.A = name, raw.astype(np.float32), A
        self.lag, self.lag_pre = int(lag), int(lag_pre)
        self.lag_len = (self.lag_pre + out_len) if self.lag else 0
        self.context = self.lag + self.lag_pre if self.lag else in_len
        # v0.5 multi-scale profile: the last n_chunks*chunk steps averaged into n_chunks values
        # (traffic: last 24 h as 24 hourly means; rainfall: last 360 days as 12 monthly means)
        self.profile = profile
        self.prof_len = profile[0] if profile else 0
        if profile:
            self.context = max(self.context, profile[0] * profile[1])
        self.extra_len = self.lag_len + self.prof_len
        self.period_idx, self.dow, self.period = period_idx.astype(np.int64), dow.astype(np.int64), period
        self.in_len, self.out_len, self.bounds = in_len, out_len, bounds
        self.transform, self.mask_zero, self.units, self.meta = transform, mask_zero, units, meta or {}
        self.T, self.N = raw.shape
        self.observed = ~np.isnan(self.raw)
        f = self.fwd(np.nan_to_num(self.raw, nan=0.0))
        a, b = bounds["train"]
        tr = f[:b][self.observed[:b]]
        self.mean, self.std = float(tr.mean()), float(tr.std())   # TRAIN split only
        if stats is not None:
            self.mean, self.std = float(stats[0]), float(stats[1])
        self.norm = np.where(self.observed, (f - self.mean) / self.std, 0.0).astype(np.float32)
        self.clim = None if stats is not None else self._climatology()   # none in zero-shot transfer

    def _climatology(self):
        """v0.4: per-location median of the TRAIN split at the same time of week (traffic, +-1 slot)
        or day of year (rainfall, +-7 days), in normalised target space. Shape [keys, N]."""
        _, b = self.bounds["train"]
        traffic = self.period == 288
        self.key = (self.dow * 288 + self.period_idx) if traffic else self.period_idx
        nk, w = (7 * 288, 1) if traffic else (366, 7)
        tr = np.where(self.observed[:b], self.norm[:b], np.nan)
        if self.mask_zero:
            tr = np.where(self.raw[:b] > 1e-4, tr, np.nan)
        tbl = np.zeros((nk, self.N), np.float32)
        for k in range(nk):
            rows = tr[np.isin(self.key[:b], [(k + j) % nk for j in range(-w, w + 1)])]
            if len(rows):
                with np.errstate(all="ignore"):
                    tbl[k] = np.nan_to_num(np.nanmedian(rows, 0))
        return tbl

    def fwd(self, x):
        return np.log1p(np.maximum(x, 0)) if self.transform == "log1p" else x

    def inv(self, z):   # z: normalised model output (torch) -> real units
        v = z * self.std + self.mean
        return torch.clamp(torch.expm1(v), min=0) if self.transform == "log1p" else v

    def indices(self, split, stride=1):
        a, b = self.bounds[split]
        start = a + self.in_len
        if self.lag:          # need lag + lag_pre steps of look-back; only ever trims the TRAIN start
            start = max(start, self.lag + self.lag_pre)
        if self.profile:
            start = max(start, self.profile[0] * self.profile[1])
        return np.arange(start, b - self.out_len + 1)[::stride]

    def valid(self, y):
        """Which ground-truth values count in metrics."""
        m = ~np.isnan(y)
        if self.mask_zero:
            m &= y > 1e-4
        return m

    def batch(self, idx, device="cpu"):
        hi = idx[:, None] + np.arange(-self.in_len, 0)[None]
        fo = idx[:, None] + np.arange(self.out_len)[None]
        if self.lag:          # seasonal window first, then the recent history
            lg = idx[:, None] - self.lag + np.arange(-self.lag_pre, self.out_len)[None]
            hi = np.concatenate([lg, hi], 1)
        x = torch.from_numpy(self.norm[hi]).to(device)
        obs = torch.from_numpy(self.observed[hi]).to(device)
        if self.profile:      # multi-scale profile first: [profile, seasonal window, recent history]
            nc, cl = self.profile
            pw = idx[:, None] + np.arange(-nc * cl, 0)[None]
            pv = self.norm[pw].reshape(len(idx), nc, cl, self.N)
            po = self.observed[pw].reshape(len(idx), nc, cl, self.N)
            cnt = po.sum(2)
            prof = np.where(cnt > 0, (pv * po).sum(2) / np.maximum(cnt, 1), 0.0).astype(np.float32)
            x = torch.cat([torch.from_numpy(prof).to(device), x], 1)
            obs = torch.cat([torch.from_numpy(cnt > 0).to(device), obs], 1)
        y = torch.from_numpy(self.raw[fo]).to(device)                 # real units, NaN = unknown
        p = torch.from_numpy(self.period_idx[idx - 1]).to(device)      # cycle position of last obs
        d = torch.from_numpy(self.dow[idx - 1]).to(device)
        c = None if self.clim is None else torch.from_numpy(self.clim[self.key[fo]]).to(device)
        return x, obs, y, p, d, c

    def describe(self):
        n = [len(self.indices(s)) for s in ("train", "val", "test")]
        return (f"{self.name}: {self.T} steps x {self.N} locations, "
                f"{int((self.A > 0).sum()) - self.N} links; train/val/test windows = {n}")


# --------------------------------------------------------------------------- PeMS traffic
PEMS_START = {"PEMS08": dt.date(2016, 7, 1), "PEMS04": dt.date(2018, 1, 1)}


def load_pems(name, lc=False, stats=None, profile=False):
    raw = np.load(f"{RAW}/{name}.npz")["data"][..., 0].astype(np.float32)
    T, N = raw.shape
    edges = np.atleast_2d(np.genfromtxt(f"{RAW}/{name}.csv", delimiter=",", skip_header=1))
    dist = np.full((N, N), np.inf)
    for s, t, c in edges:
        dist[int(s), int(t)] = dist[int(t), int(s)] = c
    sigma = dist[np.isfinite(dist)].std()
    A = np.exp(-(dist / sigma) ** 2)
    A[A < 0.1] = 0.0
    t = np.arange(T)
    dow = (t // 288 + PEMS_START[name].weekday()) % 7
    n_tr, n_va = int(T * 0.6), int(T * 0.2)
    bounds = {"train": (0, n_tr), "val": (n_tr, n_tr + n_va), "test": (n_tr + n_va, T)}
    return SeriesData(name, raw, _row_norm(A), t % 288, dow, 288, 12, 12, bounds,
                      transform="none", mask_zero=True, units="vehicles / 5 min",
                      lag=288 if lc else 0, lag_pre=12, stats=stats,
                      profile=(24, 12) if profile else None)


# --------------------------------------------------------------------------- IMD rainfall
def imd_cache_path(region, y0, y1):
    return f"{RAW}/IMD_rain_{region}_{y0}_{y1}.npz"


def download_imd(region="central_india", y0=2000, y1=2024):
    """Download IMD 0.25 deg daily gridded rainfall with imdlib and cache the region as .npz."""
    import imdlib as imd
    la0, la1, lo0, lo1 = IMD_REGIONS[region]
    obj = imd.load("rain", y0, y1, cache_dir=os.path.join(RAW, "imd_cache"))
    ds = obj.get_xarray()
    var = list(ds.data_vars)[0]
    da = ds[var]
    dims = {d.lower(): d for d in da.dims}
    tdim = next(dims[k] for k in dims if "time" in k or "date" in k)
    ladim = next(dims[k] for k in dims if "lat" in k)
    lodim = next(dims[k] for k in dims if "lon" in k)
    da = da.sortby(ladim).sortby(lodim)
    da = da.sel({ladim: slice(la0, la1), lodim: slice(lo0, lo1)}).transpose(tdim, ladim, lodim)
    if da.sizes[ladim] == 0 or da.sizes[lodim] == 0:
        raise RuntimeError(f"region {region} selected no IMD grid cells")
    arr = da.values.astype(np.float32)
    arr[arr < 0] = np.nan                                    # IMD fill value (-999) -> missing
    lat, lon = da[ladim].values, da[lodim].values
    dates = np.array([str(x)[:10] for x in da[tdim].values])
    np.savez_compressed(imd_cache_path(region, y0, y1), rain=arr, lat=lat, lon=lon, dates=dates)
    return imd_cache_path(region, y0, y1)


def load_imd(region="central_india", y0=2000, y1=2024, in_len=30, out_len=7, path=None, name=None,
             lc=False, stats=None, profile=False):
    path = path or imd_cache_path(region, y0, y1)
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} missing - run download_imd('{region}', {y0}, {y1}) first")
    z = np.load(path)
    arr, lat, lon, dates = z["rain"], z["lat"], z["lon"], z["dates"]
    Tn, H, W = arr.shape
    flat = arr.reshape(Tn, H * W)
    keep = np.where(np.mean(~np.isnan(flat), 0) > 0.9)[0]  # land cells with >90% coverage
    raw = flat[:, keep]
    ii, jj = np.unravel_index(keep, (H, W))
    clat, clon = lat[ii], lon[jj]
    # geographic graph: 8-neighbour cells on the 0.25 deg grid, weight decays with distance
    dlat = (clat[:, None] - clat[None]) * 111.0
    dlon = (clon[:, None] - clon[None]) * 111.0 * np.cos(np.deg2rad(clat[:, None]))
    km = np.sqrt(dlat ** 2 + dlon ** 2)
    A = np.where(km < 0.25 * 111 * 1.5, np.exp(-(km / 30.0) ** 2), 0.0)
    d = [dt.date.fromisoformat(s) for s in dates]
    doy = np.array([x.timetuple().tm_yday - 1 for x in d])
    years = np.array([x.year for x in d])
    yl = years.max()
    te0 = int(np.argmax(years >= yl - 2))
    va0 = int(np.argmax(years >= yl - 5))
    bounds = {"train": (0, va0), "val": (va0, te0), "test": (te0, Tn)}
    return SeriesData(name or f"IMD_{region}", raw, _row_norm(A), doy, np.zeros(Tn), 366, in_len, out_len,
                      bounds, transform="log1p", mask_zero=False, units="mm / day",
                      meta={"lat": clat.tolist(), "lon": clon.tolist(),
                            "years": [int(years.min()), int(yl)],
                            "test_years": [int(yl - 2), int(yl)], "val_years": [int(yl - 5), int(yl - 3)]},
                      lag=365 if lc else 0, lag_pre=0, stats=stats,
                      profile=(12, 30) if profile else None)


def load(name, lc=False, stats=None, profile=False, **kw):
    if name.startswith("PEMS"):
        return load_pems(name, lc=lc, stats=stats, profile=profile)
    if name == "SYNTHETIC_IMD":   # test-only data, never reported
        return load_imd(path=os.path.join(RAW, "SYNTHETIC_IMD_rain_test.npz"), name="SYNTHETIC_IMD",
                        lc=lc, stats=stats, profile=profile)
    if name.startswith("IMD_"):
        return load_imd(name[4:], lc=lc, stats=stats, profile=profile, **kw)
    raise ValueError(name)
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/pwm/data.py").read().encode()).hexdigest()[:16] == "13d7f947b3baef68", "typo in pwm/data.py"
print("cell 1 files written")


In [ ]:
import hashlib, os
os.makedirs("/kaggle/working/pwm_kit/", exist_ok=True)
os.makedirs("/kaggle/working/pwm_kit/data/raw", exist_ok=True)
os.makedirs("/kaggle/working/pwm_kit/pwm", exist_ok=True)
open("/kaggle/working/pwm_kit/pwm/model.py","w").write(r'''"""PWM-Dyn: the first trained component of Paradox World Model.

A spatiotemporal dynamics model over a physical network. It learns:
  * temporal dynamics   - how each location's state evolves over the last hour
  * periodic structure  - daily / weekly rhythms (time-of-day, day-of-week embeddings)
  * spatial propagation - how state flows along the REAL road graph (distance-weighted)
                          plus an optional learned "latent" connectivity
  * uncertainty         - quantile heads (10th / 50th / 90th percentile) for every forecast

Two variants:
  transductive=True  : adds per-location identity embeddings + learned adjacency (max accuracy
                       on a known region; cannot transfer to a new region by construction)
  transductive=False : region-agnostic - uses only observations, time and the physical graph,
                       so the same weights can be run on a road network it has never seen.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

QUANTILES = (0.1, 0.5, 0.9)


class GraphMix(nn.Module):
    """Two-hop diffusion over the physical graph (+ learned graph), gated residual."""

    def __init__(self, d, n_graphs, hops=2, drop=0.1):
        super().__init__()
        self.hops = hops
        self.proj = nn.Linear(d * (1 + n_graphs * hops), d)
        self.ff = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(2 * d, d))
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.drop = nn.Dropout(drop)

    def forward(self, h, graphs):  # h [B, N, d]
        outs = [h]
        for A in graphs:
            z = h
            for _ in range(self.hops):
                z = torch.einsum("nm,bmd->bnd", A, z)
                outs.append(z)
        h = self.n1(h + self.drop(self.proj(torch.cat(outs, -1))))
        return self.n2(h + self.drop(self.ff(h)))


class PWMDyn(nn.Module):
    def __init__(self, n_nodes=None, in_len=12, out_len=12, d=96, layers=3,
                 transductive=True, drop=0.1, steps_per_day=288, mask_aware=False, lag_len=0,
                 use_clim=False):
        super().__init__()
        self.transductive, self.out_len, self.mask_aware = transductive, out_len, mask_aware
        self.recent = in_len                     # v0.4: cues use the recent history only
        in_len = in_len + lag_len                # v0.4: + seasonal window (0 = v0.3 behaviour)
        k = 2 if mask_aware else 1   # v0.2+: each input step also carries an "observed?" flag
        self.temporal = nn.Sequential(nn.Linear(in_len * k, d), nn.GELU(), nn.Linear(d, d))
        self.stats = nn.Linear(3 + (1 if mask_aware else 0), d)  # last, mean, trend (+ share observed)
        self.tod = nn.Embedding(steps_per_day, d)
        self.dow = nn.Embedding(7, d)
        n_g = 1
        if transductive:
            self.node = nn.Parameter(torch.randn(n_nodes, d) * 0.1)
            self.src = nn.Parameter(torch.randn(n_nodes, 16) * 0.1)
            self.dst = nn.Parameter(torch.randn(n_nodes, 16) * 0.1)
            n_g = 2
        self.blocks = nn.ModuleList([GraphMix(d, n_g, drop=drop) for _ in range(layers)])
        self.head = nn.Linear(d, out_len * len(QUANTILES))
        self.use_clim = use_clim
        if use_clim:                              # v0.4: train-split climatology for the forecast steps
            self.clim_enc = nn.Linear(out_len, d)

    def learned_graph(self):
        return F.softmax(F.relu(self.src @ self.dst.T), dim=-1)

    def forward(self, x, tod, dow, A, obs=None, clim=None):
        # x [B, L, N] normalised history; obs [B, L, N] 1 = observed, 0 = missing
        xt = x.transpose(1, 2)                                     # [B, N, L]
        if self.mask_aware:
            o = torch.ones_like(xt) if obs is None else obs.transpose(1, 2).float()
            xt = xt * o
            xr, orr = xt[..., -self.recent:], o[..., -self.recent:]
            cues = torch.stack([xr[..., -1], (xr.sum(-1) / orr.sum(-1).clamp(min=1)),
                                xr[..., -1] - xr[..., 0], orr.mean(-1)], -1)
            h = self.temporal(torch.cat([xt, o], -1)) + self.stats(cues)
        else:
            cues = torch.stack([xt[..., -1], xt.mean(-1), xt[..., -1] - xt[..., 0]], -1)
            h = self.temporal(xt) + self.stats(cues)
        h = h + (self.tod(tod) + self.dow(dow))[:, None, :]
        ct = None
        if self.use_clim:
            ct = torch.zeros(x.shape[0], self.out_len, x.shape[2], device=x.device) if clim is None else clim
            ct = ct.transpose(1, 2)                                # [B, N, H]
            h = h + self.clim_enc(ct)
        graphs = [A]
        if self.transductive:
            h = h + self.node[None]
            graphs.append(self.learned_graph())
        for blk in self.blocks:
            h = blk(h, graphs)
        out = self.head(h).view(x.shape[0], x.shape[2], self.out_len, len(QUANTILES))
        base = xt[..., -1:]                                        # predict change from last state ...
        if self.use_clim and self.mask_aware:                      # ... or from climatology if it is dark
            seen = o[..., -1:] if obs is not None else torch.ones_like(base)
            base = seen * base + (1 - seen) * ct
        out = out + base[..., None]
        q = out.permute(0, 2, 1, 3)                                # [B, H, N, Q]
        return torch.sort(q, dim=-1).values                        # enforce q10 <= q50 <= q90


def pinball(pred, y, mask):
    """Quantile (pinball) loss; pred [B,H,N,Q], y [B,H,N] (both normalised)."""
    qs = torch.tensor(QUANTILES, device=pred.device)
    e = y[..., None] - pred
    l = torch.maximum(qs * e, (qs - 1) * e)
    m = mask[..., None].float()
    return (l * m).sum() / (m.sum() * len(QUANTILES)).clamp(min=1)
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/pwm/model.py").read().encode()).hexdigest()[:16] == "b1f5d9681d6c6347", "typo in pwm/model.py"
open("/kaggle/working/pwm_kit/pwm/calibration.py","w").write(r'''"""v0.4: split-conformal calibration of PWM's 80% interval (conformalised quantile regression,
Romano et al. 2019), fitted on the VALIDATION split only and applied unchanged to test.

Scores are computed in the model's own target space (log1p for rainfall, identity for traffic),
so the widening is multiplicative on rainfall and additive on traffic. One offset per horizon.
The median (point forecast) is never changed, so MAE/RMSE/MAPE are unaffected.
"""
import numpy as np


def _fwd(ds, v):
    return np.log1p(np.maximum(v, 0)) if ds.transform == "log1p" else v


def _inv(ds, v):
    return np.maximum(np.expm1(v), 0) if ds.transform == "log1p" else v


def fit_cqr(ds, q10, q90, y, valid, level=0.8):
    """Arrays [S, H, N] in real units (validation). Returns delta [H] in target space."""
    lo, hi, yt = _fwd(ds, q10), _fwd(ds, q90), _fwd(ds, np.nan_to_num(y))
    s = np.maximum(lo - yt, yt - hi)
    deltas = []
    for h in range(s.shape[1]):
        sh = s[:, h][valid[:, h]]
        n = len(sh)
        k = min(1.0, level * (1 + 1 / max(n, 1)))
        deltas.append(float(np.quantile(sh, k)) if n else 0.0)
    return np.array(deltas, np.float32)


def apply_cqr(ds, q10, q50, q90, delta):
    d = delta[None, :, None]
    lo = _inv(ds, _fwd(ds, q10) - d)
    hi = _inv(ds, _fwd(ds, q90) + d)
    return np.minimum(lo, q50), q50, np.maximum(hi, q50)
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/pwm/calibration.py").read().encode()).hexdigest()[:16] == "280de0e2fc695b3e", "typo in pwm/calibration.py"
open("/kaggle/working/pwm_kit/pwm/evaluation.py","w").write(r'''"""Shared scoring: every model (PWM or an opponent) is scored by this one function on
exactly the same windows, so comparisons are like-for-like."""
import numpy as np


def score(q10, q50, q90, y, valid):
    """All arrays [S, H, N] in real units. valid = which truths count."""
    e = (q50 - y)[valid]
    yt = y[valid]
    nz = yt > 1e-4
    out = {
        "MAE": float(np.abs(e).mean()),
        "RMSE": float(np.sqrt((e ** 2).mean())),
        "MAPE": float((np.abs(e[nz]) / yt[nz]).mean() * 100),
        "per_horizon_MAE": [float(np.abs((q50[:, h] - y[:, h])[valid[:, h]]).mean())
                            for h in range(y.shape[1])],
        "n_values_scored": int(valid.sum()),
    }
    if q10 is not None:
        inside = (y >= q10) & (y <= q90)
        out["interval_80_coverage"] = float(inside[valid].mean())
        out["interval_80_mean_width"] = float((q90 - q10)[valid].mean())
    return out


def simple_baselines(ds, idx):
    """Last value and seasonal historical average (from TRAIN data only)."""
    fo = idx[:, None] + np.arange(ds.out_len)[None]
    y = ds.raw[fo]
    valid = ds.valid(y)
    last = np.repeat(np.nan_to_num(ds.raw[idx - 1])[:, None], ds.out_len, 1)
    _, b = ds.bounds["train"]
    key = ds.dow[:b] * ds.period + ds.period_idx[:b]
    table = np.full((7 * ds.period, ds.N), np.nan, np.float32)
    for k in np.unique(key):
        table[k] = np.nanmean(ds.raw[:b][key == k], 0)
    table = np.where(np.isnan(table), np.nanmean(ds.raw[:b], 0), table)
    ha = table[ds.dow[fo] * ds.period + ds.period_idx[fo]]
    return {"last_value": score(None, last, None, y, valid),
            "historical_average": score(None, ha, None, y, valid)}
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/pwm/evaluation.py").read().encode()).hexdigest()[:16] == "c17ab93f091a81b1", "typo in pwm/evaluation.py"
open("/kaggle/working/pwm_kit/train.py","w").write(r'''"""Train PWM-Dyn (v0.4) from scratch (random initialisation) on real observations.

v0.4 changes vs v0.3: --lc adds the seasonal-lag window (long-context variant, Track B of ParadoxBench-1);
default recipe batch 64 / 40 epochs / peak lr 3e-3 / validation every epoch on every 4th window.

  python train.py --data PEMS08 --version v0.3-s0 --seed 0
  python train.py --data IMD_central_india --version v0.3-s0 --seed 0

Every run is saved to models/<data>/<version>/ (weights + report). A run is PROMOTED for its
dataset only if its VALIDATION MAE beats the currently promoted run. Test numbers are
recorded but never used to choose models.
"""
import argparse, json, os, sys, time
import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from pwm.data import load, ROOT
from pwm.evaluation import score
from pwm.model import PWMDyn, pinball

DEV = "cuda" if torch.cuda.is_available() else "cpu"


def build(ds, cfg):
    return PWMDyn(ds.N, in_len=ds.in_len, out_len=ds.out_len, d=cfg["d"], layers=cfg["layers"],
                  transductive=cfg["variant"] == "transductive", steps_per_day=ds.period,
                  mask_aware=True, lag_len=ds.extra_len, use_clim=cfg.get("clim", False))


@torch.no_grad()
def predict(model, ds, idx, bs=64, drop_nodes=None):
    """Returns q10, q50, q90, y as numpy [S, H, N] in real units."""
    model.eval()
    A = ds.A.to(next(model.parameters()).device)
    dev = A.device
    Q, Y = [], []
    for s in range(0, len(idx), bs):
        x, obs, y, p, d, c = ds.batch(idx[s:s + bs], dev)
        if drop_nodes is not None:                       # simulate offline sensors
            obs = obs.clone(); obs[..., drop_nodes] = False
            x = x * obs
        q = ds.inv(model(x, p, d, A, obs, c))
        Q.append(q.cpu().numpy()); Y.append(y.cpu().numpy())
    q, y = np.concatenate(Q), np.concatenate(Y)
    return q[..., 0], q[..., 1], q[..., 2], y


def evaluate(model, ds, split, stride=1):
    q10, q50, q90, y = predict(model, ds, ds.indices(split, stride))
    return score(q10, q50, q90, y, ds.valid(y))


def run(args):
    ds = load(args.data, lc=args.lc, profile=args.profile)
    print(ds.describe(), "| device:", DEV, flush=True)
    torch.manual_seed(args.seed); np.random.seed(args.seed)
    cfg = {k: getattr(args, k) for k in ("d", "layers", "variant", "clim", "lc", "profile")}
    model = build(ds, cfg).to(DEV)
    A = ds.A.to(DEV)
    nparam = sum(p.numel() for p in model.parameters())
    print(f"PWM-Dyn [{args.variant}{', long-context' if args.lc else ''}{', profile' if args.profile else ''}] context {ds.context} steps, parameters: {nparam:,} (random init, no pretrained weights)", flush=True)
    tr = ds.indices("train")
    steps = args.epochs * (len(tr) // args.bs + 1)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, args.lr, total_steps=steps)
    best, best_state, log, t0 = 1e18, None, [], time.time()
    for ep in range(args.epochs):
        model.train()
        perm, tot = np.random.permutation(tr), 0.0
        for s in range(0, len(perm), args.bs):
            x, obs, y, p, d, c = ds.batch(perm[s:s + args.bs], DEV)
            if args.mask_p > 0:          # hide random locations: forces use of physical neighbours
                rate = torch.rand(len(x), 1, 1, device=DEV) * args.mask_p
                obs = obs & (torch.rand(len(x), 1, ds.N, device=DEV) >= rate)
                x = x * obs
            valid = ~torch.isnan(y)
            if ds.mask_zero:
                valid &= y > 1e-4
            y0 = torch.nan_to_num(y, nan=0.0)
            yt = torch.log1p(y0.clamp(min=0)) if ds.transform == "log1p" else y0
            yn = (yt - ds.mean) / ds.std
            q = model(x, p, d, A, obs, c)
            loss = pinball(q, yn, valid) + (torch.abs(q[..., 1] - yn) * valid).sum() / valid.sum().clamp(min=1)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); sched.step()
            tot += loss.item() * len(x)
        val = evaluate(model, ds, "val", stride=args.val_stride)
        log.append({"epoch": ep + 1, "loss": tot / len(perm), "val_MAE": val["MAE"]})
        print(f"ep {ep+1:3d} loss {tot/len(perm):.4f} val MAE {val['MAE']:.3f} ({(time.time()-t0)/60:.1f} min)", flush=True)
        if val["MAE"] < best:
            best, best_state = val["MAE"], {k: v.detach().clone() for k, v in model.state_dict().items()}
        if args.max_minutes and (time.time() - t0) / 60 > args.max_minutes:
            print("time budget reached", flush=True)
            break
    model.load_state_dict(best_state)

    out = os.path.join(ROOT, "models", ds.name, args.version)
    os.makedirs(out, exist_ok=True)
    torch.save({k: v.cpu() for k, v in model.state_dict().items()}, f"{out}/weights.pt")
    rep = {"model": "PWM-Dyn", "provenance": "Paradox-trained from random initialisation",
           "dataset": ds.name, "version": args.version, "parameters": nparam,
           "config": vars(args), "train_log": log,
           "val": evaluate(model, ds, "val", stride=2)}
    json.dump(rep, open(f"{out}/report.json", "w"), indent=1)
    v = rep["val"]          # test numbers are NOT printed here: ParadoxBench scoring happens once, in paradoxbench/run_pwm.py
    print(f"{ds.name} {args.version}: best val MAE {v['MAE']:.3f} cov80 {v['interval_80_coverage']*100:.1f}% "
          f"({(time.time()-t0)/60:.1f} min)", flush=True)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="PEMS08")
    ap.add_argument("--version", required=True)
    ap.add_argument("--variant", default="transductive", choices=["transductive", "inductive"])
    ap.add_argument("--lc", action="store_true", help="v0.4 long-context variant (seasonal-lag window)")
    ap.add_argument("--profile", action="store_true", help="v0.5 multi-scale profile of the last day / year")
    ap.add_argument("--no_clim", dest="clim", action="store_false",
                    help="disable the train-split climatology input (used for zero-shot transfer models)")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--lr", type=float, default=3e-3)
    ap.add_argument("--d", type=int, default=96)
    ap.add_argument("--layers", type=int, default=3)
    ap.add_argument("--mask_p", type=float, default=0.3)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--val_stride", type=int, default=4)
    ap.add_argument("--max_minutes", type=float, default=0, help="stop early after this many minutes (0 = off)")
    run(ap.parse_args())
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/train.py").read().encode()).hexdigest()[:16] == "033319c007ce0305", "typo in train.py"
print("cell 2 files written")


In [ ]:
import hashlib, os
os.makedirs("/kaggle/working/pwm_kit/", exist_ok=True)
os.makedirs("/kaggle/working/pwm_kit/data/raw", exist_ok=True)
os.makedirs("/kaggle/working/pwm_kit/paradoxbench", exist_ok=True)
open("/kaggle/working/pwm_kit/paradoxbench/__init__.py","w").write(r'''''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/paradoxbench/__init__.py").read().encode()).hexdigest()[:16] == "e3b0c44298fc1c14", "typo in paradoxbench/__init__.py"
open("/kaggle/working/pwm_kit/paradoxbench/spec.py","w").write(r'''"""ParadoxBench-1 - Paradox Physical-Dynamics Benchmark, version 1. Frozen definitions.

Everything a submission is scored against is fixed here: tasks, splits, official test windows,
context budgets per track, the hidden-sensor set of the robustness track, and the metrics.
Change nothing in this file without bumping the benchmark version.
"""
import numpy as np

VERSION = "ParadoxBench-1"
STRIDE = 2            # official windows = every 2nd test window (all models, all tracks)
HIDDEN_SEED = 2026    # robustness track: sensors withheld at test time
HIDDEN_SHARE = 0.30
LEVEL = 0.80          # prediction-interval level scored (q10..q90)

# task id -> (loader name, description, units, horizon steps, standard context, open context)
TASKS = {
    "PEMS08":  ("PEMS08", "Caltrans District 8 highway flow, 170 sensors, 5-min", "vehicles / 5 min", 12, 12, 512),
    "PEMS04":  ("PEMS04", "Caltrans District 4 highway flow, 307 sensors, 5-min", "vehicles / 5 min", 12, 12, 512),
    "IMD-CI":  ("IMD_central_india", "IMD 0.25 deg daily rainfall, central India (MP)", "mm / day", 7, 30, 365),
    "IMD-WC":  ("IMD_west_coast", "IMD 0.25 deg daily rainfall, Konkan / Western Ghats", "mm / day", 7, 30, 365),
}

TRACKS = {
    "A": "Standard: look-back limited to the standard context (12 steps / 30 days); any training "
         "on the task's TRAIN split; pretrained models allowed if they respect the context limit.",
    "B": "Open context: look-back up to 512 steps (traffic) / 365 days (rainfall); pretrained "
         "foundation models compete here zero-shot.",
    "R": "Robustness: 30% of locations (fixed by HIDDEN_SEED) have NO observations in the test "
         "period; models must still forecast them. Scored on the hidden locations only.",
    "Z": "Zero-shot transfer: forecast PEMS08 with a model that has never seen any PEMS08 data "
         "(not even its normalisation statistics). Context as Track B.",
}


def official_windows(ds):
    """Indices of the official test windows. ds must be loaded WITHOUT a seasonal lag."""
    a, b = ds.bounds["test"]
    return np.arange(a + ds.in_len, b - ds.out_len + 1)[::STRIDE]


def validation_windows(ds):
    a, b = ds.bounds["val"]
    return np.arange(a + ds.in_len, b - ds.out_len + 1)[::STRIDE]


def hidden_sensors(n):
    k = int(round(HIDDEN_SHARE * n))
    return np.sort(np.random.default_rng(HIDDEN_SEED).permutation(n)[:k])
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/paradoxbench/spec.py").read().encode()).hexdigest()[:16] == "f2ae983a16d86d30", "typo in paradoxbench/spec.py"
open("/kaggle/working/pwm_kit/paradoxbench/metrics.py","w").write(r'''"""ParadoxBench-1 metrics. One function scores every submission (PWM, TimesFM, baselines).

Point:          MAE, RMSE, MAPE (non-zero truths only) of the median q50.
Probabilistic:  QL  = mean pinball loss over the three quantiles (10/50/90) - a proper score;
                      2*QL is a 3-point approximation of CRPS.
                IS80 = interval (Winkler) score of [q10, q90] at 80%: width + 10*(miss below) + 10*(miss above);
                      rewards intervals that are narrow AND cover. Lower is better.
                COV80 = share of truths inside [q10, q90] (target 80%), WIDTH = mean width.
"""
import numpy as np

TAUS = (0.1, 0.5, 0.9)


def bench_score(q10, q50, q90, y, valid):
    e = (q50 - y)[valid]
    yt = y[valid]
    nz = yt > 1e-4
    out = {"MAE": float(np.abs(e).mean()), "RMSE": float(np.sqrt((e ** 2).mean())),
           "MAPE": float((np.abs(e[nz]) / yt[nz]).mean() * 100) if nz.any() else None,
           "per_horizon_MAE": [float(np.abs((q50[:, h] - y[:, h])[valid[:, h]]).mean())
                               for h in range(y.shape[1])],
           "n_values": int(valid.sum())}
    if q10 is None:
        return out
    lo, hi = q10[valid], q90[valid]
    ql = 0.0
    for tau, q in zip(TAUS, (lo, q50[valid], hi)):
        d = yt - q
        ql += np.maximum(tau * d, (tau - 1) * d).mean()
    a = 0.2
    is80 = (hi - lo) + (2 / a) * np.maximum(lo - yt, 0) + (2 / a) * np.maximum(yt - hi, 0)
    out.update({"QL": float(ql / 3), "IS80": float(is80.mean()),
                "COV80": float(((yt >= lo) & (yt <= hi)).mean()), "WIDTH": float((hi - lo).mean())})
    return out
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/paradoxbench/metrics.py").read().encode()).hexdigest()[:16] == "760c70f34d0b6e7b", "typo in paradoxbench/metrics.py"
open("/kaggle/working/pwm_kit/paradoxbench/score.py","w").write(r'''"""Score a ParadoxBench-1 submission and record it on the leaderboard.

A submission is an .npz with arrays q10, q50, q90 of shape [W, H, N] (official test windows x
horizon x locations, real units), plus a JSON-able meta dict. Anyone can submit:

  python -m paradoxbench.score --task PEMS08 --track B --sub my_model.npz --model "My model" \
      --type foundation --params 200e6 --context 512 --trained-on "pretraining corpus X"
"""
import argparse, hashlib, json, os, re, sys

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.dirname(HERE))
from pwm.data import load, ROOT
from paradoxbench.spec import TASKS, VERSION, STRIDE, official_windows, hidden_sensors
from paradoxbench.metrics import bench_score

RES = os.path.join(ROOT, "results", "paradoxbench")
SUB = os.path.join(ROOT, "submissions")


def task_data(task):
    return load(TASKS[task][0])


def truth(task, track, ds=None):
    ds = ds or task_data(task)
    idx = official_windows(ds)
    y = ds.raw[idx[:, None] + np.arange(ds.out_len)[None]]
    valid = ds.valid(y)
    if track == "R":
        keep = np.zeros(ds.N, bool)
        keep[hidden_sensors(ds.N)] = True
        valid &= keep[None, None, :]
    return idx, y, valid


def slug(s):
    return re.sub(r"[^A-Za-z0-9.+-]+", "_", s).strip("_")


def record(task, track, model, q10, q50, q90, meta, ds=None, save_submission=True):
    idx, y, valid = truth(task, track, ds)
    for name, q in (("q50", q50), ("q10", q10), ("q90", q90)):
        if q is not None and q.shape != y.shape:
            raise ValueError(f"{name} has shape {q.shape}; ParadoxBench-1 {task} expects {y.shape}")
    m = bench_score(q10, q50, q90, y, valid)
    os.makedirs(RES, exist_ok=True)
    out = {"benchmark": VERSION, "task": task, "track": track, "model": model,
           "windows": {"split": "test", "stride": STRIDE, "count": int(len(idx))},
           "metrics": m, **meta}
    if save_submission and q10 is not None:
        os.makedirs(SUB, exist_ok=True)
        p = os.path.join(SUB, f"{task}__{track}__{slug(model)}.npz")
        np.savez_compressed(p, q10=q10.astype(np.float32), q50=q50.astype(np.float32),
                            q90=q90.astype(np.float32))
        out["submission_sha256"] = hashlib.sha256(open(p, "rb").read()).hexdigest()
    json.dump(out, open(os.path.join(RES, f"{task}__{track}__{slug(model)}.json"), "w"), indent=1)
    cov = f"COV80 {m['COV80']*100:.1f}% IS80 {m['IS80']:.2f} QL {m['QL']:.3f}" if "IS80" in m else "point only"
    print(f"ParadoxBench {task}/{track} {model}: MAE {m['MAE']:.3f} RMSE {m['RMSE']:.3f} {cov}", flush=True)
    return out


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", required=True, choices=list(TASKS))
    ap.add_argument("--track", required=True, choices=list("ABRZ"))
    ap.add_argument("--sub", required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--type", default="submitted")
    ap.add_argument("--params", type=float, default=None)
    ap.add_argument("--context", type=int, default=None)
    ap.add_argument("--trained-on", default="")
    a = ap.parse_args()
    z = np.load(a.sub)
    record(a.task, a.track, a.model, z["q10"], z["q50"], z["q90"],
           {"type": a.type, "params": a.params, "context_steps": a.context, "trained_on": a.trained_on})
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/paradoxbench/score.py").read().encode()).hexdigest()[:16] == "3afdd63ce182b858", "typo in paradoxbench/score.py"
open("/kaggle/working/pwm_kit/paradoxbench/baselines.py","w").write(r'''"""ParadoxBench-1 reference baselines, all fitted on the TRAIN split only.

  last value        point only, Tracks A/B
  seasonal naive    value one season earlier (yesterday same time / same day last year), Track B
  climatology       per-location empirical 10/50/90% quantiles of the training data at the same
                    time of week (traffic, +-1 slot) or day of year (rainfall, +-7 days); Tracks A/B/R
"""
import os, sys
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from paradoxbench.spec import TASKS
from paradoxbench.score import task_data, truth, record


def climatology(ds, idx):
    _, b = ds.bounds["train"]
    traffic = ds.period == 288
    key = (ds.dow * 288 + ds.period_idx) if traffic else ds.period_idx
    nk, w = (7 * 288, 1) if traffic else (366, 7)
    tr = ds.raw[:b]
    tbl = np.zeros((3, nk, ds.N), np.float32)
    for k in range(nk):
        near = [(k + j) % nk for j in range(-w, w + 1)]
        rows = tr[np.isin(key[:b], near)]
        if ds.mask_zero:
            rows = np.where(rows > 1e-4, rows, np.nan)
        tbl[:, k] = np.nan_to_num(np.nanquantile(rows, (0.1, 0.5, 0.9), axis=0)) if len(rows) else 0
    fo = idx[:, None] + np.arange(ds.out_len)[None]
    q = tbl[:, key[fo]]
    return q[0], q[1], q[2]


def run(task):
    ds = task_data(task)
    idx, y, _ = truth(task, "A", ds)
    H = ds.out_len
    meta = {"type": "baseline", "params": 0, "trained_on": f"{task} train split"}
    last = np.repeat(np.nan_to_num(ds.raw[idx - 1])[:, None], H, 1)
    seas_lag = 288 if ds.period == 288 else 365
    seas = np.nan_to_num(ds.raw[idx[:, None] - seas_lag + np.arange(H)[None]])
    c10, c50, c90 = climatology(ds, idx)
    for tr in ("A", "B"):
        record(task, tr, "Last value", None, last, None, {**meta, "context_steps": 1}, ds)
        record(task, tr, "Climatology quantiles", c10, c50, c90, {**meta, "context_steps": 0}, ds,
               save_submission=False)
    record(task, "B", "Seasonal naive", None, seas, None, {**meta, "context_steps": seas_lag + H}, ds)
    record(task, "R", "Climatology quantiles", c10, c50, c90, {**meta, "context_steps": 0}, ds,
           save_submission=False)


if __name__ == "__main__":
    for t in (sys.argv[1:] or list(TASKS)):
        run(t)
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/paradoxbench/baselines.py").read().encode()).hexdigest()[:16] == "2164d15acfb8c0e1", "typo in paradoxbench/baselines.py"
open("/kaggle/working/pwm_kit/paradoxbench/run_pwm.py","w").write(r'''"""Score a PWM-Dyn v0.4 seed ensemble on ParadoxBench-1 (one task, one track), with validation-fitted
conformal calibration. Test data is touched exactly once, here.

  python paradoxbench/run_pwm.py --task PEMS08 --track B --versions v04-lc-s0 v04-lc-s1 v04-lc-s2
  python paradoxbench/run_pwm.py --task PEMS08 --track Z --source PEMS04 --versions v04-ind-s0 ...
"""
import argparse, json, os, sys
import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.dirname(HERE))
from pwm.data import load, ROOT
from pwm.calibration import fit_cqr, apply_cqr
from paradoxbench.spec import TASKS, official_windows, validation_windows, hidden_sensors
from paradoxbench.metrics import bench_score
from paradoxbench.score import record
from train import build, predict, DEV


def members(dsname, versions):
    out = []
    for v in versions:
        d = os.path.join(ROOT, "models", dsname, v)
        rep = json.load(open(f"{d}/report.json"))
        out.append((v, rep))
    return out


def ensemble(ds, mem, srcname, idx, drop=None):
    qs = []
    for v, rep in mem:
        m = build(ds, rep["config"])
        sd = torch.load(os.path.join(ROOT, "models", srcname, v, "weights.pt"), map_location="cpu")
        m.load_state_dict(sd)
        q10, q50, q90, y = predict(m.to(DEV).eval(), ds, idx, drop_nodes=drop)
        qs.append(np.stack([q10, q50, q90]))
    q = np.sort(np.mean(qs, 0), axis=0)
    return q[0], q[1], q[2], y, sum(p.numel() for p in m.parameters())


def main(a):
    loader = TASKS[a.task][0]
    src = TASKS[a.source][0] if a.track == "Z" else loader
    mem = members(src, a.versions)
    cfg = mem[0][1]["config"]
    lc, prof = bool(cfg.get("lc")), bool(cfg.get("profile"))
    assert all(bool(r["config"].get("lc")) == lc for _, r in mem), "mixed ensemble"
    base = load(loader)                                   # lag-free: defines the official windows
    test_idx, val_idx = official_windows(base), validation_windows(base)
    if a.track == "Z":                                    # no target statistics, no target calibration
        sds = load(src, lc=lc, profile=prof)
        ds = load(loader, lc=lc, stats=(sds.mean, sds.std), profile=prof)
    else:
        sds = ds = load(loader, lc=lc, profile=prof)
    drop = hidden_sensors(ds.N) if a.track == "R" else None

    # calibration on the validation split (source validation split for zero-shot transfer)
    cal_idx = validation_windows(load(src)) if a.track == "Z" else val_idx
    v10, v50, v90, vy, _ = ensemble(sds, mem, src, cal_idx, drop=None if a.track == "Z" else drop)
    vvalid = sds.valid(vy)
    if a.track == "R":
        keep = np.zeros(ds.N, bool); keep[drop] = True
        vvalid &= keep[None, None, :]
    delta = fit_cqr(sds, v10, v90, vy, vvalid)

    q10, q50, q90, y, nparam = ensemble(ds, mem, src, test_idx, drop=drop)
    c10, c50, c90 = apply_cqr(ds, q10, q50, q90, delta)
    tag = f"PWM-Dyn {a.tag}{'-LC' if lc else ''}"
    name = {"A": tag, "B": tag, "R": tag,
            "Z": f"{tag} inductive (trained on {a.source} only)"}[a.track]
    name += f" ({len(mem)}-seed, CQR)"
    from paradoxbench.score import truth
    _, yt, valid = truth(a.task, a.track, base)
    raw = bench_score(q10, q50, q90, yt, valid)
    record(a.task, a.track, name, c10, c50, c90,
           {"type": "specialist (Paradox, trained from scratch)", "params": nparam,
            "context_steps": ds.context, "members": [v for v, _ in mem],
            "trained_on": f"{a.source if a.track == 'Z' else a.task} train split only",
            "calibration": {"method": "split-conformal CQR on validation", "delta_per_horizon": delta.tolist()},
            "uncalibrated": {k: raw[k] for k in ("MAE", "IS80", "QL", "COV80", "WIDTH")}}, base)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", required=True, choices=list(TASKS))
    ap.add_argument("--track", required=True, choices=list("ABRZ"))
    ap.add_argument("--versions", nargs="+", required=True)
    ap.add_argument("--source", default="PEMS04")
    ap.add_argument("--tag", default="v0.4", help="model version label shown on the leaderboard")
    main(ap.parse_args())
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/paradoxbench/run_pwm.py").read().encode()).hexdigest()[:16] == "a42d96a872107872", "typo in paradoxbench/run_pwm.py"
open("/kaggle/working/pwm_kit/paradoxbench/leaderboard.py","w").write(r'''"""Build LEADERBOARD.md from results/paradoxbench/*.json (every row scored by paradoxbench.metrics on the official
windows). Rows are sorted by MAE within each task and track."""
import glob, json, os, sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from pwm.data import ROOT
from paradoxbench.spec import TASKS, TRACKS, VERSION, STRIDE


def fmt(v, f="{:.2f}"):
    return "-" if v is None else f.format(v)


def main():
    rows = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(ROOT, "results", "paradoxbench", "*.json")))]
    L = [f"# {VERSION} leaderboard", "",
         f"Official windows: every {STRIDE}nd test window. Lower is better for MAE, RMSE, QL and IS80; "
         "COV80 should be close to 80%.", ""]
    for task in TASKS:
        for tr in "ABRZ":
            rs = sorted([r for r in rows if r["task"] == task and r["track"] == tr], key=lambda r: r["metrics"]["MAE"])
            if not rs:
                continue
            L += [f"## {task} - Track {tr}", "", f"_{TRACKS[tr]}_", "",
                  "| # | Model | Type | Params | Context | MAE | RMSE | MAPE | QL | IS80 | COV80 |",
                  "|---|---|---|---|---|---|---|---|---|---|---|"]
            for i, r in enumerate(rs, 1):
                m = r["metrics"]
                p = r.get("params")
                ps = "-" if p is None else (f"{p/1e6:.1f}M" if p >= 1e5 else str(int(p)))
                L.append(f"| {i} | {r['model']} | {r.get('type','')} | {ps} | {r.get('context_steps','-')} | "
                         f"{fmt(m['MAE'])} | {fmt(m['RMSE'])} | {fmt(m.get('MAPE'), '{:.1f}%')} | "
                         f"{fmt(m.get('QL'), '{:.3f}')} | {fmt(m.get('IS80'))} | "
                         f"{fmt(m.get('COV80') and m['COV80']*100, '{:.1f}%')} |")
            L.append("")
    open(os.path.join(ROOT, "LEADERBOARD.md"), "w").write("\n".join(L))
    print("\n".join(L))
    json.dump(rows, open(os.path.join(ROOT, "results", "paradoxbench_all.json"), "w"))


if __name__ == "__main__":
    main()
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/paradoxbench/leaderboard.py").read().encode()).hexdigest()[:16] == "c49cf9c503bac8cd", "typo in paradoxbench/leaderboard.py"
open("/kaggle/working/pwm_kit/check_provenance.py","w").write(r'''"""Proves PWM contains nothing copied from external models.

Checks:
 1. Every saved PWM checkpoint contains exactly PWM-Dyn's own parameters (same names, same shapes,
    same count as a freshly built PWM-Dyn) and nothing else.
 2. PWM's own code (pwm/, train.py, evaluate_pwm.py) imports no external model library
    (timesfm, transformers, huggingface_hub, openai, ...).
 3. Opponent code lives only in opponents/ and never writes into models/.
Writes PROVENANCE_REPORT.json with a SHA-256 fingerprint of every PWM checkpoint.
"""
import ast, glob, hashlib, json, os, sys
import torch

ROOT = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, ROOT)
from pwm.data import load
from train import build

FORBIDDEN = {"timesfm", "timesfm3", "transformers", "huggingface_hub", "openai", "groq",
             "chronos", "moirai", "peft", "safetensors", "diffusers", "timm"}
PWM_CODE = glob.glob(f"{ROOT}/pwm/*.py") + glob.glob(f"{ROOT}/paradoxbench/*.py") + [f"{ROOT}/train.py"]


def imports(path):
    tree = ast.parse(open(path).read())
    out = set()
    for n in ast.walk(tree):
        if isinstance(n, ast.Import):
            out |= {a.name.split(".")[0] for a in n.names}
        elif isinstance(n, ast.ImportFrom) and n.module and n.level == 0:
            out.add(n.module.split(".")[0])
    return out


def main():
    rep, ok = {"checkpoints": [], "code": {}}, True
    for f in PWM_CODE:
        bad = imports(f) & FORBIDDEN
        rep["code"][os.path.relpath(f, ROOT)] = "clean" if not bad else f"IMPORTS {sorted(bad)}"
        ok &= not bad
    datasets = {}
    for w in sorted(glob.glob(f"{ROOT}/models/*/*/weights.pt")):
        ds_name, version = w.split(os.sep)[-3], w.split(os.sep)[-2]
        cfg = json.load(open(os.path.join(os.path.dirname(w), "report.json")))["config"]
        key = (ds_name, bool(cfg.get("lc")), bool(cfg.get("profile")))
        if key not in datasets:
            datasets[key] = load(ds_name, lc=key[1], profile=key[2])
        fresh = build(datasets[key], cfg).state_dict()
        sd = torch.load(w, map_location="cpu")
        same_keys = set(sd) == set(fresh)
        same_shapes = same_keys and all(sd[k].shape == fresh[k].shape for k in sd)
        n = sum(v.numel() for v in sd.values())
        good = same_keys and same_shapes and n == sum(v.numel() for v in fresh.values())
        ok &= good
        rep["checkpoints"].append({"dataset": ds_name, "version": version, "parameters": n,
                                   "only_pwm_parameters": good,
                                   "sha256": hashlib.sha256(open(w, "rb").read()).hexdigest()})
    stray = [p for p in glob.glob(f"{ROOT}/models/**/*", recursive=True)
             if os.path.isfile(p) and os.path.basename(p) not in ("weights.pt", "report.json")]
    rep["unexpected_files_in_models"] = [os.path.relpath(p, ROOT) for p in stray]
    ok &= not stray
    rep["verdict"] = ("PASS: PWM checkpoints contain only PWM-Dyn's own trained parameters; "
                      "PWM code imports no external model library") if ok else "FAIL - see details"
    json.dump(rep, open(f"{ROOT}/PROVENANCE_REPORT.json", "w"), indent=1)
    print(json.dumps(rep, indent=1))
    sys.exit(0 if ok else 1)


if __name__ == "__main__":
    main()
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/check_provenance.py").read().encode()).hexdigest()[:16] == "58b225cb86fedd94", "typo in check_provenance.py"
print("cell 3 files written")


In [ ]:
import hashlib, os
os.makedirs("/kaggle/working/pwm_kit/", exist_ok=True)
os.makedirs("/kaggle/working/pwm_kit/data/raw", exist_ok=True)
open("/kaggle/working/pwm_kit/run_v05.py","w").write(r'''"""PWM-Dyn v0.5: validation sweep, then full 3-seed training and ParadoxBench-1 scoring, on 2 GPUs.

Stage 1 - sweep (seed 0, long-context variant) on one traffic task (PEMS08) and one rainfall task (IMD-CI);
          the configuration with the lowest VALIDATION MAE is kept per domain.
Stage 2 - 3 seeds x 4 tasks x {standard (Track A), long-context (Track B/R)} with the chosen configuration,
          plus 3 inductive seeds on PEMS04 for zero-shot transfer (Track Z).
Stage 3 - score every track; write LEADERBOARD.md; provenance audit.
TimesFM is not re-run: its ParadoxBench-1 results are recorded by the v0.4 study on the same windows.

  python run_v05.py            # full
  python run_v05.py --quick    # pipeline check
"""
import argparse, json, os, queue, subprocess, sys, threading, time

ROOT = os.path.dirname(os.path.abspath(__file__))
PY = sys.executable
LOADER = {"PEMS08": "PEMS08", "PEMS04": "PEMS04", "IMD-CI": "IMD_central_india", "IMD-WC": "IMD_west_coast"}
DOMAIN = {"PEMS08": "traffic", "PEMS04": "traffic", "IMD-CI": "rain", "IMD-WC": "rain"}
SWEEP_TASK = {"traffic": "PEMS08", "rain": "IMD-CI"}
CONFIGS = {
    "e40":            [],
    "e100":           ["--epochs", "100"],
    "e100-prof":      ["--epochs", "100", "--profile"],
    "e100-d128":      ["--epochs", "100", "--d", "128"],
    "e100-prof-d128": ["--epochs", "100", "--profile", "--d", "128"],
}
KEEP = ("ParadoxBench ", "best val", "Error", "Traceback", "error:", "time budget")
LOCK = threading.Lock()


def sh(lane, args, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    p = subprocess.Popen([PY] + args, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in p.stdout:
        tail = (tail + [line.rstrip()])[-12:]
        if any(k in line for k in KEEP):
            with LOCK:
                print(f"[{lane} gpu{gpu} {time.strftime('%H:%M:%S')}] {line.rstrip()}", flush=True)
    p.wait()
    if p.returncode:
        with LOCK:
            print(f"[{lane}] FAILED ({p.returncode}): {' '.join(args)}\n" + "\n".join(tail), flush=True)


def run_parallel(jobs, n_gpu):
    """jobs: list of (lane, args). One worker per GPU pulls from a shared queue."""
    q = queue.Queue()
    for j in jobs:
        q.put(j)

    def worker(g):
        while True:
            try:
                lane, args = q.get_nowait()
            except queue.Empty:
                return
            sh(lane, args, g)
    ts = [threading.Thread(target=worker, args=(g,)) for g in range(max(n_gpu, 1))]
    [t.start() for t in ts]
    [t.join() for t in ts]


def val(task, version):
    f = os.path.join(ROOT, "models", LOADER[task], version, "report.json")
    return json.load(open(f))["val"]["MAE"] if os.path.exists(f) else float("inf")


def main(a):
    try:
        import torch
        n_gpu = torch.cuda.device_count()
    except Exception:
        n_gpu = 0
    quick = ["--epochs", "1", "--max_minutes", "1"] if a.quick else []
    configs = {k: (v if not a.quick else []) for k, v in CONFIGS.items()}
    if a.quick:
        configs = {"e40": [], "e100-prof": ["--profile"]}
    seeds = [0] if a.quick else [0, 1, 2]
    t0 = time.time()
    print(f"v0.5 study, GPUs: {n_gpu}", flush=True)

    # ---- stage 1: validation sweep ------------------------------------------------------
    jobs = [("sweep", ["train.py", "--data", LOADER[t], "--version", f"sw-{c}", "--seed", "0", "--lc"] + args + quick)
            for dom, t in SWEEP_TASK.items() if t in a.tasks for c, args in configs.items()]
    run_parallel(jobs, n_gpu)
    chosen = {}
    for dom, t in SWEEP_TASK.items():
        if t not in a.tasks:
            continue
        scores = {c: val(t, f"sw-{c}") for c in configs}
        chosen[dom] = min(scores, key=scores.get)
        print(f"[sweep] {dom} ({t}) validation MAE: " + ", ".join(f"{c} {v:.3f}" for c, v in scores.items())
              + f" -> chosen {chosen[dom]}", flush=True)

    # ---- stage 2: full training ---------------------------------------------------------
    jobs = []
    for t in a.tasks:
        cfg = configs[chosen[DOMAIN[t]]]
        std_cfg = [x for x in cfg if x != "--profile"]                      # Track A: standard context only
        for s in seeds:
            jobs.append(("pwm", ["train.py", "--data", LOADER[t], "--version", f"v05-s{s}", "--seed", str(s)] + std_cfg + quick))
            jobs.append(("pwm", ["train.py", "--data", LOADER[t], "--version", f"v05-lc-s{s}", "--seed", str(s), "--lc"] + cfg + quick))
    if "PEMS04" in a.tasks and "PEMS08" in a.tasks:
        cfg = configs[chosen["traffic"]]
        for s in seeds:
            jobs.append(("pwm", ["train.py", "--data", "PEMS04", "--version", f"v05-ind-s{s}", "--seed", str(s), "--lc",
                                 "--variant", "inductive", "--no_clim"] + cfg + quick))
    run_parallel(jobs, n_gpu)

    # ---- stage 3: score ------------------------------------------------------------------
    jobs = []
    for t in a.tasks:
        std, lc = [f"v05-s{s}" for s in seeds], [f"v05-lc-s{s}" for s in seeds]
        vs, vl = sum(val(t, v) for v in std) / len(std), sum(val(t, v) for v in lc) / len(lc)
        best = lc if vl <= vs else std
        print(f"[pwm] {t}: val MAE std {vs:.3f} | long-context {vl:.3f} -> Track B/R use "
              f"{'long-context' if best is lc else 'standard'}", flush=True)
        for tr, vers in (("A", std), ("B", best), ("R", best)):
            jobs.append(("score", ["paradoxbench/run_pwm.py", "--task", t, "--track", tr, "--tag", "v0.5", "--versions"] + vers))
    if "PEMS04" in a.tasks and "PEMS08" in a.tasks:
        jobs.append(("score", ["paradoxbench/run_pwm.py", "--task", "PEMS08", "--track", "Z", "--source", "PEMS04",
                               "--tag", "v0.5", "--versions"] + [f"v05-ind-s{s}" for s in seeds]))
    jobs.append(("score", ["paradoxbench/baselines.py"] + a.tasks))
    run_parallel(jobs, n_gpu)
    print(f"=== v0.5 done in {(time.time()-t0)/60:.1f} min ===", flush=True)
    subprocess.run([PY, "paradoxbench/leaderboard.py"], cwd=ROOT)
    subprocess.run([PY, "check_provenance.py"], cwd=ROOT, capture_output=True, text=True)
    print("PROVENANCE:", json.load(open(os.path.join(ROOT, "PROVENANCE_REPORT.json")))["verdict"])


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--quick", action="store_true")
    ap.add_argument("--tasks", nargs="+", default=list(LOADER))
    main(ap.parse_args())
''')
assert hashlib.sha256(open("/kaggle/working/pwm_kit/run_v05.py").read().encode()).hexdigest()[:16] == "88dada19569b4fab", "typo in run_v05.py"
print("cell 4 files written")


In [ ]:
%cd /kaggle/working/pwm_kit
!pip install -q imdlib timesfm 2>&1 | tail -1
!python -c "import timesfm, timesfm3, torch; print('timesfm OK, GPUs:', torch.cuda.device_count())"
!for d in PEMS08 PEMS04; do for e in npz csv; do curl -sL -o data/raw/$d.$e https://raw.githubusercontent.com/guoshnBJTU/ASTGNN/main/data/$d/$d.$e; done; done; ls -la data/raw
from pwm.data import download_imd
for r in ("central_india", "west_coast"): print("IMD:", download_imd(r, 2000, 2024))
import os
!python run_v05.py 2>&1
print("KAGGLE-RUN-DONE")
